<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Automatización, FastAPI, Git/GitHub, Makefile y proyecto final integrador**

## Del análisis local al producto reproducible

Este laboratorio cierra el curso integrando gran parte de lo trabajado anteriormente.

La idea central es:

> **un proyecto profesional debe poder ejecutarse, validarse, automatizarse, compartirse y exponerse de forma reproducible.**

Trabajaremos con:

- automatización mediante scripts;
- funciones `main()`;
- argumentos de línea de comandos;
- configuración externa;
- Makefile;
- FastAPI;
- Pydantic;
- endpoints;
- validación;
- tests de API;
- estructura de repositorio;
- Git;
- GitHub;
- `.gitignore`;
- README;
- CI con GitHub Actions;
- proyecto final de principio a fin.

## Objetivos de aprendizaje

Al finalizar deberías poder:

1. Convertir un notebook en un flujo ejecutable.
2. Crear un script con `main()`.
3. Utilizar argumentos de línea de comandos.
4. Separar configuración y lógica.
5. Automatizar un pipeline.
6. Crear y entender un Makefile.
7. Encadenar tareas `make install`, `make test`, `make quality`, `make pipeline` y `make api`.
8. Comprender las limitaciones de `make` en Windows.
9. Crear una API mínima con FastAPI.
10. Definir modelos de entrada con Pydantic.
11. Validar peticiones.
12. Crear endpoints GET y POST.
13. Testear endpoints.
14. Diseñar una estructura de repositorio profesional.
15. Crear un README reproducible.
16. Entender el flujo básico de Git.
17. Preparar un repositorio para GitHub.
18. Entender qué automatiza CI.
19. Integrar limpieza, análisis, calidad y API en un proyecto final.

## 1. Preparación

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import textwrap

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

print("Python:", sys.version)
print("Sistema operativo:", os.name)

## 2. Crear estructura del proyecto final

In [ ]:
raiz = Path("proyecto_final_demo")

if raiz.exists():
    shutil.rmtree(raiz)

carpetas = [
    raiz / "data" / "raw",
    raiz / "data" / "processed",
    raiz / "src" / "proyecto_final",
    raiz / "tests",
    raiz / "scripts",
    raiz / "config",
    raiz / ".github" / "workflows"
]

for carpeta in carpetas:
    carpeta.mkdir(
        parents=True,
        exist_ok=True
    )

(raiz / "src" / "proyecto_final" / "__init__.py").write_text(
    "",
    encoding="utf-8"
)

(raiz / "tests" / "__init__.py").write_text(
    "",
    encoding="utf-8"
)

print("Proyecto creado:", raiz.resolve())

## 2.1 Visualizar árbol inicial

In [ ]:
for ruta in sorted(raiz.rglob("*")):
    nivel = len(ruta.relative_to(raiz).parts) - 1
    sangria = "    " * nivel
    simbolo = "📁" if ruta.is_dir() else "📄"
    print(f"{sangria}{simbolo} {ruta.name}")

## 3. Crear dataset de ejemplo

In [ ]:
N = 50_000

ventas = pd.DataFrame({
    "venta_id": np.arange(1, N + 1),
    "cliente_id": rng.integers(1, 10_001, N),
    "region": rng.choice(
        ["Norte", "Sur", "Este", "Oeste", "Centro"],
        N
    ),
    "canal": rng.choice(
        ["web", "tienda", "partner"],
        N
    ),
    "unidades": rng.integers(1, 10, N),
    "precio": rng.uniform(5, 300, N)
})

ventas["importe"] = (
    ventas["unidades"]
    * ventas["precio"]
)

ruta_datos = (
    raiz
    / "data"
    / "raw"
    / "ventas.csv"
)

ventas.to_csv(
    ruta_datos,
    index=False
)

print("Dataset creado:", ruta_datos.resolve())

## 3.1 Crear configuración externa

In [ ]:
config = {
    "input_path":
        "data/raw/ventas.csv",

    "output_path":
        "data/processed/resumen.csv",

    "threshold":
        500,

    "group_by":
        "region"
}

ruta_config = (
    raiz
    / "config"
    / "config.json"
)

ruta_config.write_text(
    json.dumps(
        config,
        indent=2
    ),
    encoding="utf-8"
)

print(
    ruta_config.read_text(
        encoding="utf-8"
    )
)

## 4. Crear módulo de configuración

In [ ]:
codigo_config = '''
from pathlib import Path
import json

def cargar_configuracion(ruta):
    ruta = Path(ruta)

    if not ruta.exists():
        raise FileNotFoundError(
            f"No existe la configuración: {ruta}"
        )

    return json.loads(
        ruta.read_text(
            encoding="utf-8"
        )
    )
'''

(raiz / "src" / "proyecto_final" / "config.py").write_text(
    textwrap.dedent(codigo_config),
    encoding="utf-8"
)

## 4.1 Crear módulo de validación

In [ ]:
codigo_validation = '''
import pandas as pd

class DatosInvalidosError(Exception):
    pass

def validar_ventas(datos: pd.DataFrame) -> None:
    requeridas = {
        "venta_id",
        "cliente_id",
        "region",
        "canal",
        "unidades",
        "precio",
        "importe"
    }

    faltantes = requeridas - set(datos.columns)

    if faltantes:
        raise DatosInvalidosError(
            f"Faltan columnas: {sorted(faltantes)}"
        )

    if (datos["unidades"] <= 0).any():
        raise DatosInvalidosError(
            "Las unidades deben ser positivas."
        )

    if (datos["precio"] < 0).any():
        raise DatosInvalidosError(
            "El precio no puede ser negativo."
        )
'''

(raiz / "src" / "proyecto_final" / "validation.py").write_text(
    textwrap.dedent(codigo_validation),
    encoding="utf-8"
)

## 4.2 Crear módulo de análisis

In [ ]:
codigo_analysis = '''
import pandas as pd

def filtrar_ventas(
    datos: pd.DataFrame,
    threshold: float
) -> pd.DataFrame:

    return datos.loc[
        datos["importe"] >= threshold
    ].copy()


def resumir_ventas(
    datos: pd.DataFrame,
    group_by: str
) -> pd.DataFrame:

    return (
        datos.groupby(
            group_by,
            as_index=False
        )
        .agg(
            facturacion=("importe", "sum"),
            operaciones=("venta_id", "size"),
            clientes=("cliente_id", "nunique"),
            ticket_medio=("importe", "mean")
        )
    )
'''

(raiz / "src" / "proyecto_final" / "analysis.py").write_text(
    textwrap.dedent(codigo_analysis),
    encoding="utf-8"
)

## 4.3 Crear pipeline reusable

In [ ]:
codigo_pipeline = '''
from pathlib import Path

import pandas as pd

from .analysis import filtrar_ventas, resumir_ventas
from .validation import validar_ventas


def ejecutar_pipeline(
    input_path,
    output_path,
    threshold,
    group_by
):
    datos = pd.read_csv(input_path)

    validar_ventas(datos)

    filtrados = filtrar_ventas(
        datos,
        threshold
    )

    resumen = resumir_ventas(
        filtrados,
        group_by
    )

    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    resumen.to_csv(
        output_path,
        index=False
    )

    return resumen
'''

(raiz / "src" / "proyecto_final" / "pipeline.py").write_text(
    textwrap.dedent(codigo_pipeline),
    encoding="utf-8"
)

## 5. Crear script automatizable

In [ ]:
codigo_script = '''
from pathlib import Path
import sys

ROOT = (
    Path(__file__)
    .resolve()
    .parents[1]
)

SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(
        0,
        str(SRC)
    )

from proyecto_final.config import cargar_configuracion
from proyecto_final.pipeline import ejecutar_pipeline


def main():
    config = cargar_configuracion(
        ROOT / "config" / "config.json"
    )

    resumen = ejecutar_pipeline(
        ROOT / config["input_path"],
        ROOT / config["output_path"],
        config["threshold"],
        config["group_by"]
    )

    print(resumen)


if __name__ == "__main__":
    main()
'''

ruta_script = (
    raiz
    / "scripts"
    / "run_pipeline.py"
)

ruta_script.write_text(
    textwrap.dedent(codigo_script),
    encoding="utf-8"
)

print(
    ruta_script.read_text(
        encoding="utf-8"
    )
)

### Interpretación

El pipeline ya no depende del notebook.

Puede ejecutarse desde terminal, Makefile o CI.

## 5.1 Ejecutar script

In [ ]:
resultado = subprocess.run(
    [
        sys.executable,
        "scripts/run_pipeline.py"
    ],
    cwd=raiz,
    capture_output=True,
    text=True
)

print(resultado.stdout)
print(resultado.stderr)
print("Return code:", resultado.returncode)

## 5.2 Validar output

In [ ]:
ruta_output = (
    raiz
    / "data"
    / "processed"
    / "resumen.csv"
)

print(
    "Existe:",
    ruta_output.exists()
)

if ruta_output.exists():
    display(
        pd.read_csv(
            ruta_output
        )
    )

## 6. Automatización con argumentos CLI

Un script profesional suele necesitar parámetros.

Ejemplo:

```bash
python scripts/run_pipeline_cli.py --threshold 1000 --group-by canal
```

In [ ]:
codigo_cli = '''
from pathlib import Path
import argparse
import sys

ROOT = (
    Path(__file__)
    .resolve()
    .parents[1]
)

SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(
        0,
        str(SRC)
    )

from proyecto_final.pipeline import ejecutar_pipeline


def construir_parser():
    parser = argparse.ArgumentParser(
        description="Ejecuta pipeline de ventas"
    )

    parser.add_argument(
        "--input",
        default="data/raw/ventas.csv"
    )

    parser.add_argument(
        "--output",
        default="data/processed/resumen_cli.csv"
    )

    parser.add_argument(
        "--threshold",
        type=float,
        default=500
    )

    parser.add_argument(
        "--group-by",
        default="region"
    )

    return parser


def main():
    parser = construir_parser()
    args = parser.parse_args()

    resumen = ejecutar_pipeline(
        ROOT / args.input,
        ROOT / args.output,
        args.threshold,
        args.group_by
    )

    print(resumen)


if __name__ == "__main__":
    main()
'''

ruta_cli = (
    raiz
    / "scripts"
    / "run_pipeline_cli.py"
)

ruta_cli.write_text(
    textwrap.dedent(codigo_cli),
    encoding="utf-8"
)

## 6.1 Ejecutar CLI

In [ ]:
resultado_cli = subprocess.run(
    [
        sys.executable,
        "scripts/run_pipeline_cli.py",
        "--threshold",
        "1000",
        "--group-by",
        "canal"
    ],
    cwd=raiz,
    capture_output=True,
    text=True
)

print(
    resultado_cli.stdout
)

## 7. Crear `pyproject.toml`

In [ ]:
pyproject = '''
[project]
name = "proyecto-final"
version = "0.1.0"
description = "Proyecto integrador del curso"
requires-python = ">=3.11"

dependencies = [
    "pandas>=2.2",
    "numpy>=1.26",
    "fastapi>=0.115",
    "uvicorn>=0.30",
    "pydantic>=2.0"
]

[project.optional-dependencies]
dev = [
    "pytest>=8.0",
    "httpx>=0.27",
    "ruff>=0.6",
    "black>=24.0",
    "mypy>=1.10"
]

[build-system]
requires = ["setuptools>=69"]
build-backend = "setuptools.build_meta"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]

[tool.ruff]
line-length = 100

[tool.black]
line-length = 100
'''

(raiz / "pyproject.toml").write_text(
    textwrap.dedent(pyproject).strip()
    + "\n",
    encoding="utf-8"
)

print(
    (raiz / "pyproject.toml").read_text(
        encoding="utf-8"
    )
)

## 7.1 Crear script de calidad

In [ ]:
codigo_quality = '''
import shutil
import subprocess

comandos = [
    ["pytest", "-q"],
    ["ruff", "check", "."],
    ["black", "--check", "."],
    ["mypy", "src"],
]

for comando in comandos:
    if shutil.which(comando[0]) is None:
        print(
            f"OMITIDO: {comando[0]} no instalado"
        )
        continue

    print(
        "$",
        " ".join(comando)
    )

    resultado = subprocess.run(
        comando,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)

    if resultado.returncode != 0:
        print(resultado.stderr)
        raise SystemExit(
            resultado.returncode
        )

print("Calidad superada.")
'''

(raiz / "check_quality.py").write_text(
    textwrap.dedent(codigo_quality),
    encoding="utf-8"
)

## 8. Makefile: automatización declarativa

Un **Makefile** permite asociar nombres cortos a tareas repetitivas del proyecto.

Ejemplos:

```bash
make install
make test
make quality
make pipeline
make api
```

Esto evita recordar comandos largos y estandariza la forma de trabajar en equipo.

## 8.1 Crear Makefile

In [ ]:
makefile = '''
.PHONY: help install install-dev test lint format format-check typecheck quality pipeline pipeline-cli api clean

PYTHON ?= python

help:
	@echo "Comandos disponibles:"
	@echo "  make install       - Instala el proyecto"
	@echo "  make install-dev   - Instala dependencias de desarrollo"
	@echo "  make test          - Ejecuta tests"
	@echo "  make lint          - Ejecuta Ruff"
	@echo "  make format        - Formatea con Black"
	@echo "  make format-check  - Comprueba formato"
	@echo "  make typecheck     - Ejecuta mypy"
	@echo "  make quality       - Ejecuta todas las comprobaciones"
	@echo "  make pipeline      - Ejecuta pipeline"
	@echo "  make pipeline-cli  - Ejecuta pipeline CLI"
	@echo "  make api           - Arranca FastAPI"
	@echo "  make clean         - Elimina cachés"

install:
	$(PYTHON) -m pip install -e .

install-dev:
	$(PYTHON) -m pip install -e ".[dev]"

test:
	pytest -q

lint:
	ruff check .

format:
	black .

format-check:
	black --check .

typecheck:
	mypy src

quality: test lint format-check typecheck

pipeline:
	$(PYTHON) scripts/run_pipeline.py

pipeline-cli:
	$(PYTHON) scripts/run_pipeline_cli.py --threshold 1000 --group-by canal

api:
	uvicorn proyecto_final.api:app --reload --app-dir src

clean:
	$(PYTHON) -c "import shutil, pathlib; [shutil.rmtree(p, ignore_errors=True) for p in pathlib.Path('.').rglob('__pycache__')]"
'''

ruta_makefile = (
    raiz
    / "Makefile"
)

ruta_makefile.write_text(
    textwrap.dedent(makefile).lstrip(),
    encoding="utf-8"
)

print(
    ruta_makefile.read_text(
        encoding="utf-8"
    )
)

### Interpretación

El Makefile centraliza tareas repetitivas.

Además, `quality` depende de:

```text
test
lint
format-check
typecheck
```

Por tanto, un solo comando ejecuta varias comprobaciones.

## 8.2 Detectar si `make` está disponible

In [ ]:
make_disponible = (
    shutil.which("make")
    is not None
)

print(
    "make disponible:",
    make_disponible
)

### Nota para Windows

En Windows, `make` no suele venir instalado por defecto.

Opciones habituales:

- Git Bash + make;
- Chocolatey;
- Scoop;
- WSL;
- usar el script Python `check_quality.py` como alternativa multiplataforma.

La idea pedagógica importante es la **automatización de tareas**, no depender obligatoriamente de una herramienta concreta.

## 8.3 Ejecutar `make help`

In [ ]:
if make_disponible:

    resultado_make_help = subprocess.run(
        [
            "make",
            "help"
        ],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(
        resultado_make_help.stdout
    )

    print(
        resultado_make_help.stderr
    )

else:
    print(
        "make no está disponible en este entorno."
    )

## 8.4 Ejecutar `make pipeline`

In [ ]:
if make_disponible:

    resultado_make_pipeline = subprocess.run(
        [
            "make",
            "pipeline"
        ],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(
        resultado_make_pipeline.stdout
    )

    print(
        "Return code:",
        resultado_make_pipeline.returncode
    )

else:
    print(
        "Ejecuta directamente:"
    )

    print(
        f"{sys.executable} scripts/run_pipeline.py"
    )

## 8.5 Provocar un target inexistente

In [ ]:
if make_disponible:

    resultado_make_error = subprocess.run(
        [
            "make",
            "target_que_no_existe"
        ],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(
        resultado_make_error.stdout
    )

    print(
        resultado_make_error.stderr
    )

    print(
        "Return code:",
        resultado_make_error.returncode
    )

### Interpretación

Un target inexistente produce un fallo explícito.

Esto también es útil en automatización: un error devuelve un código de salida distinto de cero.

## 8.6 Comparar script Python y Makefile

In [ ]:
automatizacion = pd.DataFrame({
    "herramienta": [
        "Script Python",
        "Makefile"
    ],
    "ventaja": [
        "Multiplataforma y lógica compleja",
        "Comandos cortos y dependencias entre tareas"
    ],
    "uso": [
        "check_quality.py",
        "make quality"
    ]
})

automatizacion

### Interpretación

No son excluyentes.

Un Makefile puede llamar a scripts Python, y los scripts Python pueden contener la lógica compleja.

## 9. Introducción a FastAPI

FastAPI permite exponer funcionalidad Python mediante HTTP.

Un endpoint puede convertir nuestro análisis en un servicio consumible por:

- una web;
- otra aplicación;
- un dashboard;
- un pipeline externo.

## 9.1 Detectar FastAPI

In [ ]:
try:
    import fastapi
    from fastapi import FastAPI
    FASTAPI_DISPONIBLE = True
    print(
        "FastAPI:",
        fastapi.__version__
    )
except ImportError:
    FASTAPI_DISPONIBLE = False
    print(
        "FastAPI no instalado."
    )

## 9.2 Crear API mínima

In [ ]:
codigo_api = '''
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(
    title="Proyecto Final API",
    version="0.1.0"
)


class VentaInput(BaseModel):
    unidades: int = Field(
        gt=0
    )

    precio: float = Field(
        ge=0
    )

    descuento: float = Field(
        default=0.0,
        ge=0,
        le=1
    )


@app.get("/")
def root():
    return {
        "status": "ok",
        "service": "proyecto-final"
    }


@app.get("/health")
def health():
    return {
        "status": "healthy"
    }


@app.post("/calcular-importe")
def calcular_importe(
    venta: VentaInput
):
    importe = (
        venta.unidades
        * venta.precio
        * (
            1
            - venta.descuento
        )
    )

    return {
        "importe":
            importe
    }
'''

ruta_api = (
    raiz
    / "src"
    / "proyecto_final"
    / "api.py"
)

ruta_api.write_text(
    textwrap.dedent(codigo_api),
    encoding="utf-8"
)

print(
    ruta_api.read_text(
        encoding="utf-8"
    )
)

## 9.3 Ejecutar API

Desde la raíz del proyecto:

```bash
uvicorn proyecto_final.api:app --reload --app-dir src
```

o, si `make` está disponible:

```bash
make api
```

Documentación automática:

```text
http://127.0.0.1:8000/docs
```

## 9.4 Provocar validación Pydantic

In [ ]:
if FASTAPI_DISPONIBLE:
    from pydantic import BaseModel, Field

    class VentaDemo(BaseModel):
        unidades: int = Field(
            gt=0
        )
        precio: float = Field(
            ge=0
        )
        descuento: float = Field(
            default=0,
            ge=0,
            le=1
        )

    try:
        VentaDemo(
            unidades=-1,
            precio=20
        )

    except Exception as error:
        print(
            type(error).__name__
        )
        print(error)

## 10. Tests de API

In [ ]:
codigo_test_api = '''
import pytest

fastapi = pytest.importorskip(
    "fastapi"
)

from fastapi.testclient import TestClient
from proyecto_final.api import app

client = TestClient(app)


def test_health():
    response = client.get(
        "/health"
    )

    assert response.status_code == 200

    assert response.json() == {
        "status":
            "healthy"
    }


def test_calcular_importe():
    response = client.post(
        "/calcular-importe",
        json={
            "unidades": 2,
            "precio": 100,
            "descuento": 0.2
        }
    )

    assert response.status_code == 200

    assert response.json()[
        "importe"
    ] == 160


def test_validacion_unidades():
    response = client.post(
        "/calcular-importe",
        json={
            "unidades": -1,
            "precio": 100
        }
    )

    assert response.status_code == 422
'''

(raiz / "tests" / "test_api.py").write_text(
    textwrap.dedent(codigo_test_api),
    encoding="utf-8"
)

## 10.1 Tests del pipeline

In [ ]:
codigo_test_pipeline = '''
import pandas as pd

from proyecto_final.pipeline import ejecutar_pipeline


def test_pipeline(tmp_path):
    datos = pd.DataFrame({
        "venta_id": [1, 2],
        "cliente_id": [10, 20],
        "region": ["Norte", "Sur"],
        "canal": ["web", "tienda"],
        "unidades": [2, 5],
        "precio": [100, 300],
        "importe": [200, 1500]
    })

    entrada = tmp_path / "ventas.csv"
    salida = tmp_path / "resumen.csv"

    datos.to_csv(
        entrada,
        index=False
    )

    resultado = ejecutar_pipeline(
        entrada,
        salida,
        500,
        "region"
    )

    assert salida.exists()
    assert len(resultado) == 1

    assert resultado.iloc[0][
        "facturacion"
    ] == 1500
'''

(raiz / "tests" / "test_pipeline.py").write_text(
    textwrap.dedent(codigo_test_pipeline),
    encoding="utf-8"
)

## 10.2 Ejecutar pytest si está disponible

In [ ]:
pytest_disponible = (
    shutil.which("pytest")
    is not None
)

if pytest_disponible:

    env = dict(
        os.environ
    )

    env["PYTHONPATH"] = str(
        (
            raiz
            / "src"
        ).resolve()
    )

    resultado_tests = subprocess.run(
        [
            "pytest",
            "-q"
        ],
        cwd=raiz,
        capture_output=True,
        text=True,
        env=env
    )

    print(resultado_tests.stdout)
    print(resultado_tests.stderr)

else:
    print(
        "pytest no disponible."
    )

## 11. Crear `.gitignore`

In [ ]:
gitignore = '''
.venv/
__pycache__/
*.pyc
.ipynb_checkpoints/
.env
dist/
build/
*.egg-info/
.pytest_cache/
.mypy_cache/
.ruff_cache/
data/processed/*
!data/processed/.gitkeep
'''

(raiz / ".gitignore").write_text(
    textwrap.dedent(gitignore).strip()
    + "\n",
    encoding="utf-8"
)

print(
    (raiz / ".gitignore").read_text(
        encoding="utf-8"
    )
)

## 11.1 Introducción a Git

Flujo mínimo:

```bash
git init
git status
git add .
git commit -m "Initial project structure"
```

Después de conectar GitHub:

```bash
git remote add origin <URL_DEL_REPOSITORIO>
git branch -M main
git push -u origin main
```

Durante el desarrollo:

```bash
git checkout -b feature/nueva-funcionalidad
git add .
git commit -m "Add new feature"
git push
```

## 11.2 Detectar Git

In [ ]:
git_disponible = (
    shutil.which("git")
    is not None
)

print(
    "Git disponible:",
    git_disponible
)

## 11.3 Inicializar repositorio local

In [ ]:
if git_disponible:

    resultado_git = subprocess.run(
        [
            "git",
            "init"
        ],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(
        resultado_git.stdout
    )

else:
    print(
        "Git no está disponible."
    )

## 11.4 Consultar estado

In [ ]:
if git_disponible:

    resultado_git_status = subprocess.run(
        [
            "git",
            "status",
            "--short"
        ],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(
        resultado_git_status.stdout
    )

### Interpretación

`git status` permite saber:

- qué archivos no están versionados;
- qué archivos han cambiado;
- qué está preparado para commit.

## 12. Crear README reproducible

In [ ]:
readme = '''
# Proyecto Final Demo

## Objetivo

Pipeline reproducible de análisis de ventas con API FastAPI.

## Instalación

```bash
python -m venv .venv
```

### Windows

```powershell
.\\.venv\\Scripts\\Activate.ps1
```

### Linux/macOS

```bash
source .venv/bin/activate
```

Instalación:

```bash
python -m pip install -e ".[dev]"
```

## Ejecutar pipeline

```bash
python scripts/run_pipeline.py
```

o:

```bash
make pipeline
```

## Ejecutar con parámetros

```bash
python scripts/run_pipeline_cli.py --threshold 1000 --group-by canal
```

o:

```bash
make pipeline-cli
```

## Ejecutar API

```bash
uvicorn proyecto_final.api:app --reload --app-dir src
```

o:

```bash
make api
```

## Tests

```bash
pytest -q
```

o:

```bash
make test
```

## Calidad

```bash
make quality
```

Alternativa multiplataforma:

```bash
python check_quality.py
```

## Estructura

```text
data/
src/
tests/
scripts/
config/
.github/workflows/
```
'''

(raiz / "README.md").write_text(
    textwrap.dedent(readme).strip()
    + "\n",
    encoding="utf-8"
)

## 13. Integración continua: concepto

Una pipeline de CI puede ejecutar automáticamente:

```text
push / pull request
        ↓
instalar Python
        ↓
instalar dependencias
        ↓
pytest
        ↓
ruff
        ↓
black --check
        ↓
mypy
```

La calidad deja de depender de que alguien recuerde ejecutar manualmente cada comprobación.

## 13.1 Crear workflow de GitHub Actions

In [ ]:
workflow = '''
name: quality

on:
  push:
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - run: python -m pip install -e ".[dev]"

      - run: pytest -q

      - run: ruff check .

      - run: black --check .

      - run: mypy src
'''

ruta_workflow = (
    raiz
    / ".github"
    / "workflows"
    / "quality.yml"
)

ruta_workflow.write_text(
    textwrap.dedent(workflow).strip()
    + "\n",
    encoding="utf-8"
)

print(
    ruta_workflow.read_text(
        encoding="utf-8"
    )
)

### Relación Makefile ↔ CI

También podríamos reducir el workflow a:

```yaml
- run: make quality
```

si el entorno de CI dispone de `make`.

Esto centraliza la definición de tareas.

## 14. Validar estructura final

In [ ]:
obligatorios = [
    raiz / "README.md",
    raiz / "pyproject.toml",
    raiz / ".gitignore",
    raiz / "Makefile",
    raiz / "config" / "config.json",
    raiz / "src" / "proyecto_final" / "config.py",
    raiz / "src" / "proyecto_final" / "validation.py",
    raiz / "src" / "proyecto_final" / "analysis.py",
    raiz / "src" / "proyecto_final" / "pipeline.py",
    raiz / "src" / "proyecto_final" / "api.py",
    raiz / "scripts" / "run_pipeline.py",
    raiz / "scripts" / "run_pipeline_cli.py",
    raiz / "tests" / "test_pipeline.py",
    raiz / "tests" / "test_api.py",
    raiz / ".github" / "workflows" / "quality.yml"
]

estado = pd.DataFrame({
    "ruta": [
        str(
            ruta.relative_to(
                raiz
            )
        )
        for ruta in obligatorios
    ],
    "existe": [
        ruta.exists()
        for ruta in obligatorios
    ]
})

estado

## 14.1 Provocar inconsistencia

In [ ]:
ruta_makefile_backup = (
    ruta_makefile.read_text(
        encoding="utf-8"
    )
)

ruta_makefile.unlink()

print(
    "Makefile existe:",
    ruta_makefile.exists()
)

## 14.2 Detectar inconsistencia

In [ ]:
faltantes = [
    ruta
    for ruta in obligatorios
    if not ruta.exists()
]

[
    str(
        ruta.relative_to(
            raiz
        )
    )
    for ruta in faltantes
]

## 14.3 Corregir y volver a validar

In [ ]:
ruta_makefile.write_text(
    ruta_makefile_backup,
    encoding="utf-8"
)

print(
    "Faltantes restantes:",
    [
        str(
            ruta.relative_to(
                raiz
            )
        )
        for ruta in obligatorios
        if not ruta.exists()
    ]
)

### Interpretación

Incluso la automatización debe formar parte de la estructura validada del proyecto.

## 15. Visualizar árbol final

In [ ]:
for ruta in sorted(
    raiz.rglob("*")
):
    nivel = (
        len(
            ruta.relative_to(
                raiz
            ).parts
        )
        - 1
    )

    sangria = "    " * nivel
    simbolo = "📁" if ruta.is_dir() else "📄"

    print(
        f"{sangria}{simbolo} {ruta.name}"
    )

## 15.1 Matriz de responsabilidades

In [ ]:
responsabilidades = pd.DataFrame({
    "componente": [
        "config.py",
        "validation.py",
        "analysis.py",
        "pipeline.py",
        "run_pipeline.py",
        "run_pipeline_cli.py",
        "Makefile",
        "api.py",
        "tests/",
        "pyproject.toml",
        "README.md",
        "Git",
        "GitHub Actions"
    ],
    "responsabilidad": [
        "Configuración",
        "Validación",
        "Lógica analítica",
        "Orquestación",
        "Ejecución automática",
        "Ejecución parametrizada",
        "Automatización de tareas",
        "Exposición HTTP",
        "Verificación",
        "Dependencias/configuración",
        "Reproducibilidad",
        "Versionado",
        "Calidad automática"
    ]
})

responsabilidades

## 16. Flujo profesional completo

El proyecto final queda así:

```text
CSV
 ↓
configuración
 ↓
validación
 ↓
filtrado
 ↓
agregación
 ↓
exportación
 ↓
script / CLI
 ↓
Makefile
 ↓
API
 ↓
tests
 ↓
calidad
 ↓
Git
 ↓
GitHub Actions
```

La automatización deja de ser una capa opcional y pasa a formar parte del diseño del proyecto.

## 17. Mini-práctica integrada

Construye un proyecto para un caso de analítica o predicción que incluya:

```text
src/
tests/
scripts/
config/
data/
Makefile
.github/workflows/
```

Debe tener:

- configuración externa;
- validación;
- feature engineering;
- análisis;
- CLI;
- Makefile;
- FastAPI;
- tests;
- linting;
- type hints;
- README;
- Git;
- CI conceptual.

## 18. Checklist de proyecto final

- [ ] ¿La lógica está fuera del notebook?
- [ ] ¿Existe `main()`?
- [ ] ¿Hay ejecución CLI?
- [ ] ¿La configuración está separada?
- [ ] ¿Se validan datos?
- [ ] ¿Existe pipeline reusable?
- [ ] ¿Existe Makefile?
- [ ] ¿`make quality` agrupa comprobaciones?
- [ ] ¿Existe alternativa multiplataforma?
- [ ] ¿Existe API?
- [ ] ¿Las entradas de API se validan?
- [ ] ¿Hay tests de API?
- [ ] ¿Hay tests de pipeline?
- [ ] ¿Existe `pyproject.toml`?
- [ ] ¿Existe README?
- [ ] ¿Existe `.gitignore`?
- [ ] ¿El repo puede inicializar Git?
- [ ] ¿Existe workflow de calidad?
- [ ] ¿El proyecto puede reproducirse desde cero?

## 19. Ejercicios finales

### Ejercicio 1 — Automatización
Convierte un notebook en un script `main.py`.

### Ejercicio 2 — CLI
Añade al menos tres argumentos con `argparse`.

### Ejercicio 3 — Configuración
Mueve parámetros hardcodeados a JSON.

### Ejercicio 4 — Validación
Añade reglas de calidad al pipeline.

### Ejercicio 5 — Makefile
Crea targets `install`, `test`, `quality`, `pipeline` y `api`.

### Ejercicio 6 — Dependencias entre targets
Haz que `quality` dependa de `test`, `lint`, `format-check` y `typecheck`.

### Ejercicio 7 — Error Makefile
Provoca un target inexistente e interpreta el código de salida.

### Ejercicio 8 — Windows
Documenta una alternativa cuando `make` no esté instalado.

### Ejercicio 9 — FastAPI
Crea endpoint GET `/health`.

### Ejercicio 10 — POST
Crea un endpoint que reciba datos y devuelva un cálculo.

### Ejercicio 11 — Pydantic
Añade restricciones de rango.

### Ejercicio 12 — API test
Testea status code y respuesta.

### Ejercicio 13 — Error API
Provoca una validación 422.

### Ejercicio 14 — README
Documenta instalación, Makefile y ejecución.

### Ejercicio 15 — Git
Inicializa un repositorio y revisa `git status`.

### Ejercicio 16 — gitignore
Evita versionar entorno, secretos y outputs.

### Ejercicio 17 — CI
Crea un workflow que ejecute pytest, Ruff, Black y mypy.

### Ejercicio 18 — Reto final
Construye un proyecto completo que integre:

1. estructura `src/`;
2. configuración;
3. datos;
4. validación;
5. limpieza;
6. análisis;
7. optimización;
8. logging;
9. tests;
10. CLI;
11. Makefile;
12. FastAPI;
13. README;
14. Git;
15. GitHub Actions;
16. ejecución reproducible;
17. validación de estructura;
18. una demo final funcional.

## 20. Preguntas de reflexión

1. ¿Qué diferencia existe entre notebook y pipeline?
2. ¿Qué aporta `main()`?
3. ¿Por qué una CLI mejora reproducibilidad?
4. ¿Qué aporta un Makefile?
5. ¿Por qué Makefile y scripts Python se complementan?
6. ¿Qué problema puede aparecer con `make` en Windows?
7. ¿Qué aporta FastAPI?
8. ¿Qué ventaja tiene validar inputs?
9. ¿Qué diferencia hay entre GET y POST?
10. ¿Por qué testear endpoints?
11. ¿Qué aporta README?
12. ¿Qué no debe versionarse?
13. ¿Qué aporta Git?
14. ¿Qué aporta GitHub respecto a Git?
15. ¿Qué problema resuelve CI?
16. ¿Por qué separar lógica y API?
17. ¿Por qué separar configuración y código?
18. ¿Qué hace que un proyecto sea realmente reproducible?